# Finsheild — Training Notebook (Colab)

This is the Finsheild training environment. Run cells top-to-bottom in a Colab GPU/CPU runtime.

What this notebook does:
1. Detect hardware (Python, GPU availability)
2. Clone or pull the Finsheild repo
3. Install dependencies (Colab-friendly pins)
4. Mount Google Drive for persistent checkpoints/results
5. Acquire the dataset (Kaggle creds via Colab Secrets, or synthetic fallback)
6. Run training (`python -m finsheild.train`)
7. Save checkpoints + final model + metrics to Drive
8. Evaluate on the held-out test split

If a required resource is missing, the relevant cell fails loudly with a clear message — do NOT silently continue.

LightGBM trains fast on the full 284k dataset (~30s on Colab CPU). GPU is optional; this model family is CPU-native.

In [ ]:
# Cell 1 — Hardware detection
import sys, platform
print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
try:
    import torch
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("No GPU — training will run on CPU (sufficient for this tabular model)")
except Exception as e:
    print(f"Torch check skipped: {e}")
import os
print(f"IN_COLAB: {'google.colab' in sys.modules}")
print(f"Working dir: {os.getcwd()}")

In [ ]:
# Cell 2 — Clone or pull the Finsheild repo
# Set REPO_URL and REPO_BRANCH below. Leave REPO_URL empty to use a local upload (advanced).
import os, subprocess, sys
REPO_URL = ""   # e.g. "https://github.com/<you>/Finsheild.git"
REPO_BRANCH = "main"
REPO_DIR = "/content/Finsheild"

if REPO_URL:
    if os.path.isdir(REPO_DIR):
        print(f"Updating existing checkout at {REPO_DIR}")
        subprocess.check_call(["git", "-C", REPO_DIR, "fetch", "--all"])
        subprocess.check_call(["git", "-C", REPO_DIR, "checkout", REPO_BRANCH])
        subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--ff-only"])
    else:
        subprocess.check_call(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR])
    os.chdir(REPO_DIR)
    print(f"cwd: {os.getcwd()}")
else:
    print("REPO_URL not set — assuming the repo is already on the Colab runtime (e.g. uploaded as a zip).")
    if not os.getcwd().endswith("Finsheild"):
        print(f"WARNING: cwd is {os.getcwd()}; the pipeline expects to be run from the repo root.")
    REPO_DIR = os.getcwd()

In [ ]:
# Cell 3 — Install dependencies
import subprocess, sys
REQ = "requirements-colab.txt"
print(f"Installing {REQ} ...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", REQ, "lightgbm"])
print("Done.")

In [ ]:
# Cell 4 — Mount Google Drive (persistent storage for checkpoints/models/results)
import os, sys
from pathlib import Path
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DRIVE_ROOT = Path("/content/drive/MyDrive/Finsheild")
if IN_COLAB:
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"Drive mounted. Artifacts will mirror to: {DRIVE_ROOT}")
else:
    print("Not in Colab runtime — Drive mount skipped.")
print(f"DRIVE_ROOT={DRIVE_ROOT}")

In [ ]:
# Cell 5 — Acquire dataset (Kaggle via Secrets, or synthetic fallback)
# Preferred: set KAGGLE_USERNAME / KAGGLE_KEY in Colab Secrets (left panel) and expose them via os.environ below.
# Fallback: synthetic CSV via scripts/download_dataset.py --synthetic (small, for smoke tests only).
import os, subprocess, sys
from pathlib import Path

# Uncomment + populate to use Kaggle creds stored in Colab Secrets:
# from google.colab import userdata
# os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
# os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

raw = Path("data/raw/creditcard.csv")
if raw.exists():
    print(f"Raw already at {raw} ({raw.stat().st_size/1e6:.1f} MB)")
else:
    print("Attempting Kaggle download via scripts/download_dataset.py ...")
    rc = subprocess.call([sys.executable, "scripts/download_dataset.py"])
    if rc != 0 or not raw.exists():
        print("Kaggle download unavailable — generating synthetic fallback.")
        subprocess.check_call([sys.executable, "scripts/download_dataset.py", "--synthetic", "--n", "20000"])

if not raw.exists():
    raise SystemExit(f"FATAL: dataset missing at {raw}. Aborting.")
print(f"Dataset ready: {raw}")

In [ ]:
# Cell 6 — Run training (writes to models/, checkpoints/, results/ under repo)
# Override MODEL / EXPERIMENT / SEED via env vars below.
import os, subprocess, sys
os.environ.setdefault("FINSHEILD_CHECKPOINTS_DIR", "/content/Finsheild/checkpoints")
os.environ.setdefault("FINSHEILD_MODELS_DIR", "/content/Finsheild/models")
os.environ.setdefault("FINSHEILD_RESULTS_DIR", "/content/Finsheild/results")

MODEL = "lightgbm"   # 'logreg' or 'lightgbm'
EXPERIMENT = "experiment_001"
RESUME = RESUME if (RESUME := False) else False  # set RESUME=True to resume from last checkpoint
SEED = 42
TARGET_FPR = 0.01

cmd = [sys.executable, "-m", "finsheild.train",
       "--model", MODEL,
       "--experiment", EXPERIMENT,
       "--seed", str(SEED),
       "--target-fpr", str(TARGET_FPR)]
if RESUME:
    cmd.append("--resume")
print("Running:", " ".join(cmd))
subprocess.check_call(cmd)

In [ ]:
# Cell 7 — Show final test metrics
import json, os
from pathlib import Path
metrics_path = Path(os.environ.get("FINSHEILD_RESULTS_DIR", "results")) / EXPERIMENT / "metrics.json"
print(f"Metrics: {metrics_path}")
metrics = json.loads(metrics_path.read_text())
for k, v in metrics.items():
    print(f"  {k}: {v}")

In [ ]:
# Cell 8 — Mirror artifacts to Google Drive (cloud, not local disk)
import shutil, os
from pathlib import Path
if not IN_COLAB:
    print("Not in Colab — skipping Drive mirror.")
else:
    paths_to_copy = [
        Path(os.environ.get("FINSHEILD_MODELS_DIR", "models")) / EXPERIMENT,
        Path(os.environ.get("FINSHEILD_RESULTS_DIR", "results")) / EXPERIMENT,
    ]
    for src in paths_to_copy:
        if not src.exists():
            print(f"skip missing {src}")
            continue
        dst = DRIVE_ROOT / src
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"mirrored {src} -> {dst}")
    (DRIVE_ROOT / "_last_run.json").write_text(json.dumps({"experiment": EXPERIMENT, "model": MODEL}, indent=2))
    print(f"Drive sync complete. Root: {DRIVE_ROOT}")

## Troubleshooting

- `ModuleNotFoundError: finsheild` — make sure `os.chdir(REPO_DIR)` ran in Cell 2. The pipeline expects to be invoked from the repo root.
- `RuntimeError: Raw dataset missing` — re-run Cell 5 with creds set in Colab Secrets, or accept the synthetic fallback for a smoke test.
- Drive mirror fails with `OSError: [Errno 5]` — Drive not mounted; re-run Cell 4.
- Training interrupted — re-run Cell 6 with `RESUME = True` to continue from `checkpoints/<experiment>/`.
- Want to inspect the trained model — use `FraudPredictor.load(...)` from `finsheild.inference`.